# Script for aspect-based sentiment analysis using PyABSA

## Importing dependencies and loading existing preprocessed data

In [ ]:
# importing dependencies and keywords
from dependencies import *
from keywords import *

In [ ]:
# loading existing preprocessed data
df = pd.read_csv(preprocessed_csv_filepath, encoding='utf-8-sig')
print(f"Loaded {len(df)} rows from {preprocessed_csv_filepath}")

## Aspect-based sentiment analysis

### Defining functions for PyABSA

In [ ]:
# reducing logging noise and ignore deprecation warnings from libraries
logging.getLogger('pyabsa').setLevel(logging.ERROR)
warnings.filterwarnings('ignore', category=DeprecationWarning)

In [ ]:
# compiling rexeg pattern for all product quality keywords
product_quality_keyword_pattern = re.compile(
    r'\b(' + '|'.join(map(re.escape, [
        kw
        for keywords in product_quality_feature_keywords.values()
        for kw in ([keywords] if isinstance(keywords, str) else keywords)
    ])) + r')\b',
    re.IGNORECASE)

# creating reverse lookup dictionary to bind keywords to the product quality feature they are indicative of
keyword_to_aspect = {}
for aspect_category, keywords in product_quality_feature_keywords.items():
    kw_list = [keywords] if isinstance(keywords, str) else keywords
    for kw in kw_list:
        keyword_to_aspect[kw.lower()] = aspect_category

# defining function to identify keyword matches in lemmatized text, returns a list of tuples with aspect and keyword
def get_aspect_from_keywords(text):
    matched = []
    seen = set()
    for kw in product_quality_keyword_pattern.findall(text):
        key = kw.lower()
        if key not in seen:
            seen.add(key)
            matched.append((keyword_to_aspect[key], kw))
    return matched

# defining function to identify original version of lemmatized keywords
def get_surface_matches(keyword, doc):
    matches = []
    keyword_tokens = keyword.lower().split() 
    if len(keyword_tokens) == 1:
        # single-word keyword: iterate tokens and match on lemma
        for token in doc:
            if token.lemma_.lower() == keyword.lower():
                matches.append((token.idx, token.idx + len(token.text), token.text))
    else:
        # multi-word keyword: check consecutive token sequences for lemma matches
        tokens = list(doc)
        for j in range(len(tokens) - len(keyword_tokens) + 1):
            span_lemmas = [tokens[j + k].lemma_.lower() for k in range(len(keyword_tokens))]
            if span_lemmas == keyword_tokens:
                start = tokens[j].idx
                end   = tokens[j + len(keyword_tokens) - 1].idx + len(tokens[j + len(keyword_tokens) - 1].text)
                surface_form = doc.text[start:end]
                matches.append((start, end, surface_form))
    return matches

In [ ]:
# precomputing spaCy docs on bert column for original keyword matching
df['spacy_doc'] = list(nlp.pipe(df['preprocessed_text_bert_pyabsa'], batch_size=256, n_process=-1))

### Running PyABSA

#### Loading potential checkpoint

In [ ]:
# as below code takes a long time when running without a CPU, checkpoints are saved every 200 iterations to avoid losing progress due to interruptions or errors
CHECKPOINT_EVERY = 200

if os.path.exists(absa_checkpoint_filepath):
    checkpoint_df = pd.read_csv(absa_checkpoint_filepath)
    results = checkpoint_df.to_dict('records')
    already_processed = set(
        (r['post_id'], r['comment_id'], r['keyword_lemmatized'])
        for r in results
    )
    print(f"Resuming from checkpoint. {len(results)} results present.")
else:
    results = []
    already_processed = set()
    print("No checkpoint found, starting fresh")

i = len(results)

#### Running script

In [ ]:
# defining classifier with English language
sentiment_classifier = APC.SentimentClassifier(
    checkpoint='english',
    auto_device=True,
    max_seq_len = 512)

In [ ]:
# determining sentiment of each aspect (product quality feature) in each post/comment
for row in tqdm(df.itertuples(index=False), total=len(df)):
    content            = row.preprocessed_text_bert_pyabsa
    content_lemmatized = row.preprocessed_text_bert_pyabsa_lemmatized
    doc                = row.spacy_doc

    if not isinstance(content_lemmatized, str) or not content_lemmatized.strip():
        continue

    for aspect_category, matched_keyword in get_aspect_from_keywords(content_lemmatized):
        surface_matches = get_surface_matches(matched_keyword, doc)

        if not surface_matches:
            continue

        for start, end, surface_form in surface_matches:
            if (row.post_id, row.comment_id, matched_keyword) in already_processed:
                continue

            marked_content = (
                content[:start] +
                f"[B-ASP]{surface_form}[E-ASP]" +
                content[end:])

            result = sentiment_classifier.predict(
                marked_content,
                pred_sentiment=True,
                print_result=False,
                ignore_error=True)

            if result is None:
                continue

            probs = result['probs'][0]
            results.append({
                'aspect':             aspect_category,
                'sentiment_label':    result['sentiment'][0],
                'prob_negative':      float(probs[0]),
                'prob_neutral':       float(probs[1]),
                'prob_positive':      float(probs[2]),
                'keyword_lemmatized': matched_keyword,
                'keyword_original':   surface_form,
                'type':               row.type,
                'text':               content,
                'post_id':            row.post_id,
                'comment_id':         row.comment_id,
                'author':             row.author,
                'subreddit':          row.subreddit,
                'created_utc':        row.created_utc,
                'time_period':        row.time_period})

            i += 1
            if i % CHECKPOINT_EVERY == 0:
                pd.DataFrame(results).to_csv(absa_checkpoint_filepath, index=False)
                tqdm.write(f"Checkpoint saved at {i} ({len(results)} results)")

### Saving results

In [ ]:
# creating df from results
absa_df = pd.DataFrame(results)

In [ ]:
# saving df
if not absa_df.empty:
    # saving as .csv
    absa_df.to_csv(
        aspect_based_sentiment_csv_filepath,
        index=False,
        encoding='utf-8-sig',
        quoting=csv.QUOTE_ALL,
        escapechar='\\',
        lineterminator='\n')

    # saving as .tsv
    absa_df.to_csv(
        aspect_based_sentiment_tsv_filepath,
        index=False,
        sep='\t',
        encoding='utf-8-sig',
        quoting=csv.QUOTE_ALL,
        escapechar='\\',
        lineterminator='\n')

    print(f"Saved {len(absa_df)} rows to CSV and TSV")
else:
    print("No entries to save")